# Phase 4 — A/B Evaluation & Statistics (Days 16–18)

Paired analysis of developer diagnostic latency: **Control** (flag + confidence) vs.
**Treatment** (prototype + similarity + justification). **CPU-only.**

- **H1:** ≥ 30% latency reduction.  **H0:** no significant difference (p > 0.05).

Expects `artifacts/eval_logs.csv` with columns:
`participant, case_id, arm, seconds, correct` (arm ∈ {control, treatment}).

### Step 0 — get the repo onto this runtime

In [ ]:
import os, subprocess
from pathlib import Path

# ── EDIT THIS if you have a GitHub remote ────────────────────────────────────
GITHUB_URL = "https://github.com/yogijoshi86/SLMProject.git"
# ─────────────────────────────────────────────────────────────────────────────

TARGET = Path("/content/SLMProject")

if TARGET.is_dir() and (TARGET / "src" / "guardrail_audit").is_dir():
    print("Repo already present — pulling latest…")
    subprocess.run(["git", "-C", str(TARGET), "pull", "--ff-only"], check=True)

elif GITHUB_URL:
    print("Cloning from GitHub…")
    subprocess.run(["git", "clone", GITHUB_URL, str(TARGET)], check=True)
    print("Cloned to", TARGET)

else:
    # ── Google Drive fallback ─────────────────────────────────────────────────
    # Mount Drive once then point DRIVE_PATH at wherever you stored the folder.
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_PATH = "/content/drive/MyDrive/SLMProject"   # adjust if needed
    if not Path(DRIVE_PATH).is_dir():
        raise FileNotFoundError(
            f"Could not find the repo at {DRIVE_PATH}. "
            "Either set GITHUB_URL above, or copy the SLMProject folder to your Drive "
            "and update DRIVE_PATH."
        )
    import shutil
    shutil.copytree(DRIVE_PATH, str(TARGET))
    print("Copied from Drive to", TARGET)

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path("/content/SLMProject")
assert (REPO_ROOT / "src" / "guardrail_audit").is_dir(), \
    f"src/guardrail_audit not found under {REPO_ROOT}. Did the previous cell succeed?"

sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)
print("Repo root:", REPO_ROOT)

In [ ]:
# Install the project + GPU/model/explainer extras.
# transformers==4.44.2: exact pin — supports Llama-3 rope_scaling, avoids is_jax_tensor conflicts.
%pip install -q -e ".[quant,explainer,dev]" \
    "torch>=2.4.0" "torchvision>=0.19.0" \
    "transformers==4.44.2" \
    "bitsandbytes>=0.45.0"

In [ ]:
# Restart the runtime so the upgraded transformers is loaded fresh.
# After restart, re-run from the HF auth cell downwards (CLONE/LOCATE/INSTALL are done).
import os
print("Restarting runtime to pick up upgraded packages...")
os.kill(os.getpid(), 9)

In [ ]:
from guardrail_audit.utils import load_config, set_seed

# colab_smoke.yaml = 500 prompts + int8 (fits a free T4). Swap to default.yaml for full runs.
CONFIG = "config/colab_smoke.yaml"
cfg = load_config(CONFIG)
set_seed(cfg.seed)
cfg

### Generate a synthetic log to demo the analysis (delete once you have real data)

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

rng = np.random.default_rng(cfg.seed)
rows = []
for participant in range(cfg.evaluation.n_participants):
    for case in range(30):
        base = rng.normal(60, 8)
        rows.append(dict(participant=participant, case_id=f"c{case:02d}", arm="control",
                         seconds=max(5, base), correct=rng.random() < 0.80))
        rows.append(dict(participant=participant, case_id=f"c{case:02d}", arm="treatment",
                         seconds=max(5, base * rng.normal(0.62, 0.08)), correct=rng.random() < 0.90))
demo = pd.DataFrame(rows)
Path(cfg.paths.eval_logs).parent.mkdir(parents=True, exist_ok=True)
demo.to_csv(cfg.paths.eval_logs, index=False)
demo.head()

### Pair on (participant, case_id) and run the tests

In [ ]:
import pandas as pd
from guardrail_audit.evaluation import analyze_diagnostics

df = pd.read_csv(cfg.paths.eval_logs)
control = df[df.arm == "control"].set_index(["participant", "case_id"])
treatment = df[df.arm == "treatment"].set_index(["participant", "case_id"])
common = control.index.intersection(treatment.index)
control, treatment = control.loc[common].sort_index(), treatment.loc[common].sort_index()

stats = analyze_diagnostics(
    control["seconds"].tolist(), treatment["seconds"].tolist(),
    control.get("correct", pd.Series(dtype=bool)).tolist() or None,
    treatment.get("correct", pd.Series(dtype=bool)).tolist() or None,
)
stats

### Visualize + verdict

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].boxplot([control["seconds"], treatment["seconds"]], labels=["Control", "Treatment"])
ax[0].set_ylabel("Diagnostic latency (s)")
ax[0].set_title(f"Reduction {stats.reduction_pct:.1f}%  (target >= 30%)")
deltas = control["seconds"].values - treatment["seconds"].values
ax[1].hist(deltas, bins=12); ax[1].axvline(0, color="k", ls="--")
ax[1].set_xlabel("Control - Treatment (s)"); ax[1].set_title(f"paired t p={stats.t_pvalue:.4f}")
plt.tight_layout(); plt.show()

verdict = "REJECT H0 (significant)" if stats.t_pvalue < 0.05 else "FAIL TO REJECT H0"
print(f"reduction={stats.reduction_pct:.1f}% | t={stats.t_statistic:.3f} p={stats.t_pvalue:.4f} -> {verdict}")
if stats.accuracy_treatment is not None:
    print(f"accuracy: control {stats.accuracy_control:.1%} | treatment {stats.accuracy_treatment:.1%} (target >= 85%)")